# Solo Enterprise for Istio, ambient, in two parts: multicluster and workload identity

Two kind clusters (`mesh1` + `mesh2`) running Solo ambient, peered over HBONE, stood up by **one script** (`./setup.sh`). This notebook is the demo on top: every command and manifest is in the cell, nothing hidden behind a script.

- **Part 1, Multicluster.** Bookinfo on both clusters, east-west gateways, peering, **global services** (`*.mesh.internal`), cross-cluster **failover** and **takeover**, and an L7 **waypoint**.
- **Part 2, Workload identity (L4/L7).** The petshop identity story on `mesh1` only: the SVID *is* the identity, L4 authorisation in ztunnel, the shared-ServiceAccount gap, **workload claims** closing it, then the agentgateway waypoint for JWT, CEL authorisation, canary routing and identity-keyed rate limiting.

**The parts are independent**, so run Part 1 or Part 2 in isolation (both only need `./setup.sh` to be green). Open the Gloo UI first (`./demo-scripts/consoles.sh`); its service graph spans both clusters.

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> After a laptop sleep, heal expired mesh certs with `./demo-scripts/wake.sh`.

In [ ]:
# Connect — contexts, the Solo istioctl build, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CLUSTER1=kind-mesh1 CLUSTER2=kind-mesh2
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "mesh1=$CLUSTER1  mesh2=$CLUSTER2  licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
$ISTIOCTL --context $CLUSTER1 multicluster check 2>&1 | grep -E "Peers Check|Gateway Check" || echo "not peered — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"


---
# Part 1 · Multicluster ambient

**The story for the customer:** run the same app in two clusters, give it one hostname, and show traffic quietly fail over to the other cluster when one goes down, with no change to the app.

`setup.sh` has already installed Solo ambient on both clusters, sharing a single root CA, so they trust each other. From here we deploy the app, join the clusters, then break things on purpose to prove the resilience.

**Cluster layout (matches the deck):**

- `mesh1` runs reviews **v1 + v2** (no stars / black stars)
- `mesh2` also runs **v3** (red stars)

So the moment traffic crosses to mesh2, you can point at the red stars on screen.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 250" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="640" height="250" rx="10" fill="#f8fafc"/> <text x="320" y="30" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Two kind clusters, ambient mesh, shared root of trust</text> <!-- mesh1 --> <rect x="30" y="55" width="250" height="150" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="155" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1  (kind)</text> <rect x="55" y="95" width="200" height="34" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="155" y="116" text-anchor="middle" font-size="12" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="55" y="140" width="200" height="30" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="155" y="159" text-anchor="middle" font-size="11.5" fill="#334155">trust domain: mesh1</text> <text x="155" y="190" text-anchor="middle" font-size="11" fill="#64748b">no workloads yet</text> <!-- mesh2 --> <rect x="360" y="55" width="250" height="150" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="485" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2  (kind)</text> <rect x="385" y="95" width="200" height="34" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="485" y="116" text-anchor="middle" font-size="12" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="385" y="140" width="200" height="30" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="485" y="159" text-anchor="middle" font-size="11.5" fill="#334155">trust domain: mesh2</text> <text x="485" y="190" text-anchor="middle" font-size="11" fill="#64748b">no workloads yet</text> <!-- shared root CA between them --> <rect x="292" y="112" width="56" height="36" rx="6" fill="#fef9c3" stroke="#ca8a04"/> <text x="320" y="128" text-anchor="middle" font-size="10" fill="#713f12">root</text> <text x="320" y="141" text-anchor="middle" font-size="10" fill="#713f12">CA</text> <line x1="280" y1="130" x2="292" y2="130" stroke="#ca8a04" stroke-width="1.5"/> <line x1="348" y1="130" x2="360" y2="130" stroke="#ca8a04" stroke-width="1.5"/> <text x="320" y="228" text-anchor="middle" font-size="11" fill="#64748b">Same root CA signs each cluster's intermediate — one root of trust, not peered yet.</text> </svg></div>

## 1.1 · Deploy the app and put it in the mesh

**What we're doing:** deploy Bookinfo into both clusters and enrol it into the ambient mesh.

**How:** two labels on the namespace, no sidecars and no restarts:

- `istio.io/dataplane-mode=ambient`: ztunnel starts capturing every pod's traffic
- `topology.istio.io/network=<cluster>`: tells istiod which cluster the pods live in, so it can route across clusters later

**What you'll see:** each namespace carrying both labels, then the deployments per cluster, mesh1 with reviews v1 + v2 and mesh2 also with v3.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX create namespace bookinfo --dry-run=client -o yaml | kubectl --context $CTX apply -f -
  kubectl --context $CTX label namespace bookinfo \
    istio.io/dataplane-mode=ambient \
    topology.istio.io/network=${CTX#kind-} --overwrite
done

# ── verify: both namespaces carry the ambient + network labels ──
echo "mesh1:"; kubectl --context $CLUSTER1 get ns bookinfo -L istio.io/dataplane-mode -L topology.istio.io/network
echo "mesh2:"; kubectl --context $CLUSTER2 get ns bookinfo -L istio.io/dataplane-mode -L topology.istio.io/network


In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
BOOK=https://raw.githubusercontent.com/istio/istio/release-1.24/samples/bookinfo/platform/kube
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo.yaml
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo-versions.yaml
done
# match the deck: mesh1 runs v1+v2 only; mesh2 keeps v3 (red stars) as well
kubectl --context $CLUSTER1 -n bookinfo delete deploy reviews-v3 --ignore-not-found

# a client pod on mesh1. It does DOUBLE duty:
#   1. a steady ~1 req/s background loop to the GLOBAL hostname, so the Gloo UI
#      service graph continuously reflects routing (mesh1 normally, shifts to
#      mesh2 during failover, back on restore — visible live, no manual traffic).
#      It is a harmless no-op until §1.4 creates reviews.bookinfo.mesh.internal.
#   2. the measurement pod: the observe cells below `kubectl exec` their own
#      curls into it for the precise podname counts (exec spawns an independent
#      process, so the loop and the measurements never interfere).
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: apps/v1
kind: Deployment
metadata: { name: demo-client, namespace: bookinfo, labels: { app: demo-client } }
spec:
  replicas: 1
  selector: { matchLabels: { app: demo-client } }
  template:
    metadata: { labels: { app: demo-client } }
    spec:
      containers:
        - name: curl
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                curl -s -o /dev/null -m 4 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null || true
                sleep 1
              done
EOF

kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s
kubectl --context $CLUSTER2 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s

# ── verify: mesh1 has reviews v1+v2 (no v3); mesh2 has v1+v2+v3 ──
echo "mesh1 deployments:"; kubectl --context $CLUSTER1 -n bookinfo get deploy
echo "mesh2 deployments:"; kubectl --context $CLUSTER2 -n bookinfo get deploy


## 1.2 · Join the two clusters into one mesh

**What we're doing:** peer the clusters so a service in one can be reached from the other.

**How:** two istioctl commands (the Solo build).

`expose` builds the east-west gateway on each cluster. It sits on a LoadBalancer IP with an HBONE listener on `:15008` (the cross-cluster mTLS tunnel real traffic flows through) and istiod's xDS on `:15012`.

`link` then points each cluster at the other's east-west address, so the two istiods discover each other's endpoints. No remote kubeconfig secrets, Solo peers istiod-to-istiod over xDS.

`setup.sh` already ran these, so they no-op here. Run them anyway to show how the clusters were joined.

**What you'll see:** on each cluster, two Gateways, the local `istio-eastwest` plus a `istio-remote-peer-<other>` pointing at the peer, and `multicluster check` reporting the clusters connected.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# east-west gateway on each cluster (idempotent)
for CTX in $CLUSTER1 $CLUSTER2; do
  $ISTIOCTL --context $CTX multicluster expose -n istio-eastwest
done
# link them bidirectionally (idempotent)
$ISTIOCTL multicluster link --namespace istio-eastwest --contexts=$CLUSTER1,$CLUSTER2


In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# what link created: the local east-west gateway (istio-eastwest) + a remote
# peer gateway (istio-remote-peer-<other>) per cluster. Class + address make the
# local-vs-remote distinction obvious.
for C in $CLUSTER1 $CLUSTER2; do
  echo "── ${C#kind-} ──"
  kubectl --context $C -n istio-eastwest get gateways \
    -o custom-columns=NAME:.metadata.name,CLASS:.spec.gatewayClassName,ADDRESS:.status.addresses[0].value
done
echo
$ISTIOCTL --context $CLUSTER1 multicluster check | grep -E "Peers Check|Gateway Check"


Now the picture: each cluster has its own east-west gateway on a LoadBalancer IP, and each points at the peer's. Traffic crosses over HBONE (`:15008`); the control planes sync over xDS (`:15012`).

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 290" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="640" height="290" rx="10" fill="#f8fafc"/> <text x="320" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Peered over HBONE — east-west gateways + link</text> <!-- mesh1 --> <rect x="20" y="50" width="240" height="170" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="140" y="74" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1</text> <rect x="42" y="88" width="196" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="140" y="108" text-anchor="middle" font-size="11.5" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="42" y="128" width="196" height="40" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="140" y="145" text-anchor="middle" font-size="12" font-weight="600" fill="#7c2d12">east-west GW (LoadBalancer)</text> <text x="140" y="160" text-anchor="middle" font-size="11" fill="#92400e">192.168.97.142</text> <rect x="42" y="178" width="196" height="28" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="140" y="196" text-anchor="middle" font-size="10.5" fill="#475569">istio-remote-peer-mesh2 →</text> <!-- mesh2 --> <rect x="380" y="50" width="240" height="170" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="500" y="74" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2</text> <rect x="402" y="88" width="196" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="500" y="108" text-anchor="middle" font-size="11.5" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="402" y="128" width="196" height="40" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="500" y="145" text-anchor="middle" font-size="12" font-weight="600" fill="#7c2d12">east-west GW (LoadBalancer)</text> <text x="500" y="160" text-anchor="middle" font-size="11" fill="#92400e">192.168.97.160</text> <rect x="402" y="178" width="196" height="28" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="500" y="196" text-anchor="middle" font-size="10.5" fill="#475569">← istio-remote-peer-mesh1</text> <!-- arrows between the two east-west GWs --> <defs> <marker id="ar" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <line x1="260" y1="134" x2="380" y2="134" stroke="#334155" stroke-width="1.6" marker-end="url(#ar)"/> <line x1="380" y1="134" x2="260" y2="134" stroke="#334155" stroke-width="1.6" marker-end="url(#ar)" opacity="0"/> <text x="320" y="128" text-anchor="middle" font-size="11" font-weight="600" fill="#0f172a">HBONE :15008</text> <text x="320" y="152" text-anchor="middle" font-size="10" fill="#475569">cross-cluster mTLS tunnel</text> <line x1="260" y1="186" x2="380" y2="186" stroke="#64748b" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#ar)"/> <text x="320" y="180" text-anchor="middle" font-size="11" font-weight="600" fill="#0f172a">xDS :15012</text> <text x="320" y="205" text-anchor="middle" font-size="10" fill="#475569">endpoint discovery</text> <text x="320" y="255" text-anchor="middle" font-size="11" fill="#64748b">expose builds each local east-west GW; link points each cluster at the peer's address.</text> <text x="320" y="272" text-anchor="middle" font-size="11" fill="#64748b">No remote kubeconfig secrets — istiod-to-istiod over xDS.</text> </svg></div>

## 1.3 · Put the app behind an agentgateway ingress

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 300" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="640" height="300" rx="10" fill="#f8fafc"/> <text x="320" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">agentgateway ingress fronts the app on mesh1</text> <defs> <marker id="ar3" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- external client --> <rect x="18" y="120" width="70" height="50" rx="8" fill="#e2e8f0" stroke="#64748b"/> <text x="53" y="141" text-anchor="middle" font-size="11" fill="#334155">client /</text> <text x="53" y="156" text-anchor="middle" font-size="11" fill="#334155">browser</text> <line x1="88" y1="145" x2="120" y2="145" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <!-- mesh1 --> <rect x="120" y="48" width="250" height="200" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="245" y="70" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1</text> <rect x="140" y="82" width="210" height="42" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="245" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">agentgateway ingress (LB)</text> <text x="245" y="115" text-anchor="middle" font-size="10.5" fill="#166534">GatewayClass enterprise-agentgateway</text> <line x1="245" y1="124" x2="245" y2="140" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <rect x="150" y="140" width="190" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="245" y="160" text-anchor="middle" font-size="12" fill="#1e293b">productpage / reviews …</text> <rect x="150" y="180" width="190" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="245" y="197" text-anchor="middle" font-size="11" font-weight="600" fill="#7c2d12">east-west GW</text> <text x="245" y="210" text-anchor="middle" font-size="10" fill="#92400e">.142</text> <!-- mesh2 --> <rect x="450" y="48" width="172" height="200" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="536" y="70" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2</text> <rect x="466" y="140" width="140" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="536" y="160" text-anchor="middle" font-size="12" fill="#1e293b">reviews (+v3) …</text> <rect x="466" y="180" width="140" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="536" y="197" text-anchor="middle" font-size="11" font-weight="600" fill="#7c2d12">east-west GW</text> <text x="536" y="210" text-anchor="middle" font-size="10" fill="#92400e">.160</text> <!-- HBONE between east-west GWs --> <line x1="340" y1="197" x2="466" y2="197" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <text x="403" y="190" text-anchor="middle" font-size="10.5" font-weight="600" fill="#0f172a">HBONE :15008</text> <text x="320" y="278" text-anchor="middle" font-size="11" fill="#64748b">One north-south front door on mesh1; the clusters stay peered over HBONE for the failover steps.</text> </svg></div>

**What we're doing:** give the app a front door on mesh1, then open it in the browser.

**How:** one `Gateway` (class `enterprise-agentgateway`, Solo's standard Gateway API data plane) and one `HTTPRoute` for productpage. It lands on a LoadBalancer IP.

**What you'll see:** the ingress address printed, an HTTP 200 from `/productpage`, and the page in your browser.

**Keep that browser tab open.** In §1.5 to §1.6 you'll watch the reviews stars break and come back, red meaning served from mesh2.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: bookinfo-gateway, namespace: bookinfo }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - { name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: productpage, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - backendRefs: [{ name: productpage, port: 9080 }]
EOF
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Programmed gateway/bookinfo-gateway --timeout=90s
# the agentgateway data-plane pod is created ON DEMAND, a few seconds AFTER the
# Gateway goes Programmed — poll for it before waiting Ready (kubectl wait errors
# on "no matching resources" if the pod does not exist yet)
for i in $(seq 1 40); do
  kubectl --context $CLUSTER1 -n bookinfo get pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway 2>/dev/null | grep -q . && break
  sleep 3
done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready \
  pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway --timeout=120s

GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo "open →  http://$GW:8080/productpage"
# LB IP + pod may need a moment to answer — retry until the first 200
for i in $(seq 1 20); do
  code=$(curl -s -o /dev/null -w '%{http_code}' -m 5 "http://$GW:8080/productpage")
  [ "$code" = "200" ] && break; sleep 3
done
echo "HTTP $code"


## 1.4 · Give the service one hostname across both clusters

**What we're doing:** make `reviews` reachable on a single name that works no matter which cluster is actually serving it. This is the foundation for the failover we prove next.

**How:** one label, `solo.io/service-scope=global`. Solo Istio publishes `reviews.bookinfo.mesh.internal`, a hostname whose endpoints span both clusters, and it leaves the original `reviews.bookinfo.svc.cluster.local` local-only, so nothing changes for callers already using it.

By default this global hostname prefers **local** endpoints and only crosses to the other cluster when the local ones are gone. That preference is what makes the next two steps a clean failover.

**What you'll see:** the auto-generated ServiceEntry for the global hostname, and every call from mesh1 still served by mesh1's own pods, because local is healthy.

From here the `demo-client` pod's steady background traffic makes the **Gloo UI Graph move on its own**. Open it now: both clusters ticked, `bookinfo` namespace, smallest time window.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo label svc reviews solo.io/service-scope=global --overwrite
done
sleep 8
# istiod generated the global hostname automatically:
kubectl --context $CLUSTER1 -n istio-system get serviceentry
$ISTIOCTL --context $CLUSTER1 multicluster check 2>/dev/null | grep -iA2 "shared services" || true


In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# mesh1 still has local reviews pods, so calls to the GLOBAL hostname stay LOCAL
echo "--- 10 calls to reviews.bookinfo.mesh.internal from mesh1 (expect only mesh1 pods) ---"
for i in $(seq 1 10); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c
echo
echo "mesh1 reviews pods:"; kubectl --context $CLUSTER1 -n bookinfo get pods -l app=reviews -o name


## 1.5 · Kill it locally and watch it fail over

**What we're doing:** take mesh1's `reviews` away completely and show the global hostname keep serving from mesh2.

**How:** scale mesh1's reviews to zero. Nothing else changes, same hostname, same callers.

**What you'll see, two hostnames diverging:**

- the **global** hostname (`...mesh.internal`) keeps answering, now from mesh2's pods including v3, over the HBONE tunnel
- the **local** hostname (`...svc.cluster.local`) breaks, because §1.4 left it local-only and there is nothing local left

In the Gloo UI Graph the demo-client edge crosses from mesh1 to mesh2. In the browser, productpage's reviews panel goes down (it uses the local hostname). §1.6 fixes exactly that.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=0; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=delete pod -l app=reviews --timeout=60s
sleep 8

echo "--- GLOBAL hostname: still serving, now from mesh2 (expect v3 pods too) ---"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c

echo
echo "--- LOCAL hostname: local-only, and local is gone ---"
kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
  curl -s -o /dev/null -w "reviews.bookinfo.svc.cluster.local -> %{http_code}\n" -m 5 \
  http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 || true
# refresh the productpage tab: the reviews panel is down (it calls the local hostname)


## 1.6 · Take over the local hostname too

**What we're doing:** make the app's own local hostname fail over as well, without touching the app. Real apps like productpage call `reviews.bookinfo.svc.cluster.local`, and you are not going to rewrite them.

**How:** one more label, `solo.io/service-takeover=true` (Solo Istio 1.27.2+). It redirects the local hostname onto the global endpoint set, so productpage's unchanged calls now reach mesh2. (`solo.io/service-scope=global-only` is the older single-label spelling of the same behaviour.)

**What you'll see:** the local hostname served cross-cluster by mesh2, and after a browser refresh the reviews panel back, now with **red stars** because mesh2's v3 is answering. That is cross-cluster traffic you can point at on the screen.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo label svc reviews solo.io/service-takeover=true --overwrite
sleep 10

echo "--- LOCAL hostname, taken over: served cross-cluster by mesh2 ---"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo
echo "refresh →  http://$GW:8080/productpage   (reviews back — red stars = mesh2's v3)"


**Gloo UI moment.** In the Graph, the `demo-client` edge that was on mesh1's reviews has now crossed into **mesh2's** reviews. That is the cross-cluster hop, drawn live by the steady background traffic, no manual curling. Drop the time window to its smallest setting so it redraws quickly.

Now restore mesh1 and watch locality reassert. Within ~30s the graph edge returns to **mesh1** and the productpage stars go black again. Traffic only crossed clusters during the outage.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod -l app=reviews --timeout=120s
sleep 12
echo "--- local endpoints are back: traffic returns LOCAL (PreferNetwork) ---"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c


## 1.7 · The same gateway does traffic management too

**What we're doing:** before we leave the ingress, show it is a full L7 gateway, not just a front door.

**How:** two plain Gateway API objects on the same ingress from §1.3:

- an `HTTPRoute` that weights `/reviews` 80/20 across v1 and v2
- a policy that rate-limits `/productpage` to 2 requests a second

**What you'll see:** roughly an 80/20 split across the two review versions in the curl counts, and the productpage requests going 200, 200, then 429 once the limit bites.

These hit the ingress routes directly, so they show in the counts below, not in the productpage panel.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-canary, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - matches: [{ path: { type: PathPrefix, value: /reviews } }]
    backendRefs:
    - { name: reviews-v1, port: 9080, weight: 80 }
    - { name: reviews-v2, port: 9080, weight: 20 }
EOF
sleep 3
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
for i in $(seq 1 20); do curl -s "http://$GW:8080/reviews/0" | grep -oE 'reviews-v[0-9]'; done | sort | uniq -c


In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayPolicy
metadata: { name: ingress-ratelimit, namespace: bookinfo }
spec:
  targetRefs:
  - { group: gateway.networking.k8s.io, kind: HTTPRoute, name: productpage }
  traffic:
    rateLimit:
      local:
      - requests: 2
        unit: Seconds
EOF
sleep 4
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
for i in $(seq 1 8); do curl -s -o /dev/null -w "%{http_code} " "http://$GW:8080/productpage"; done; echo
# → first 2 pass (200), the rest 429


## 1.8 · L7 between services: the waypoint

**What we're doing:** add L7 control for traffic between services inside the mesh, not just at the edge.

**How:** ambient adds a **waypoint**, and on Solo Enterprise the waypoint is **agentgateway** (class `enterprise-agentgateway-waypoint`), not Envoy. Enrol the namespace onto it with one label, then attach an `HTTPRoute` whose parent is the reviews **Service**, so productpage's own calls to reviews are routed with no ingress involved.

**What you'll see:** every mesh call to reviews pinned to v2, agentgateway logging each request, and in the browser reviews staying solid **black** (v2) on every refresh.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: bookinfo-waypoint
  namespace: bookinfo
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint=bookinfo-waypoint --overwrite
kubectl --context $CLUSTER1 -n bookinfo rollout status deploy/bookinfo-waypoint --timeout=150s

# route AT the waypoint: parentRef is the reviews SERVICE (GAMMA) — pin all traffic to v2
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-mesh-split, namespace: bookinfo }
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: reviews
      port: 9080
  rules:
    - backendRefs:
        - { name: reviews-v2, port: 9080, weight: 100 }
EOF
sleep 8
echo "--- 8 mesh calls to reviews (via the waypoint): all v2 ---"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]'
done | sort | uniq -c
# proof the waypoint carried it: agentgateway logs every request
kubectl --context $CLUSTER1 -n bookinfo logs deploy/bookinfo-waypoint --tail=3


## 1.R · Reset Part 1

**What we're doing:** put Bookinfo back to steady state so Part 1 can be re-run.

**How:** remove the waypoint, canary, rate limit and takeover label, and scale reviews back to two pods. This keeps the app, the ingress, the global label and demo-client.

**What you'll see:** no routes, gateways or policies left in bookinfo, and reviews back to two running pods.

For a full "square one" wipe of the whole demo (both parts), use `./demo-scripts/reset.sh` instead.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CLUSTER1 -n bookinfo delete httproute reviews-mesh-split reviews-canary --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo delete gateway bookinfo-waypoint --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo delete agentgatewaypolicy ingress-ratelimit --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo label svc reviews solo.io/service-takeover- 2>/dev/null || true
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1; done
echo "Part 1 reset."

# ── verify: no waypoint/canary/ratelimit left; reviews back to 2 local pods ──
echo "waypoint label: $(kubectl --context $CLUSTER1 get ns bookinfo -o jsonpath='{.metadata.labels.istio\.io/use-waypoint}' 2>/dev/null || echo none)"
kubectl --context $CLUSTER1 -n bookinfo get httproute,gateway,agentgatewaypolicy 2>/dev/null | grep -v NAME || echo "  (no routes/gateways/policies — clean)"
kubectl --context $CLUSTER1 -n bookinfo get pods -l app=reviews


---
# Part 2 · Workload identity: the certificate is the identity

**The story for the customer:** in ambient every workload gets a cryptographic identity from its certificate, and you write access policy against that identity, not against IPs or network rules. This part runs on `mesh1` only and needs no peering, so it stands alone.

We take one app, a petshop, through the whole model: prove identity from the certificate, authorise on it at L4 in ztunnel, hit the one thing service-account identity cannot do, close that gap with workload claims, then add an agentgateway waypoint for the L7 controls (JWT, CEL authorisation, canary, rate limiting).

The trust domain on `mesh1` is **`mesh1`** (a Helm value at install, not `cluster.local`), so identities read `spiffe://mesh1/ns/petshop/sa/<sa>`.

**The cast:**

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API — `GET /pets`, `DELETE /pets/{id}` |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

Each client curls `petstore:8080/pets` every 2 seconds and prints the HTTP code. The observe cells read the client's last log line, which is why each policy step sleeps briefly first. A denied client shows `000` (connection reset), an allowed one `200`.

In [ ]:
# Part 2 context — independent of Part 1. Run this first (after Connect).
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 2.1 · Deploy the petshop app

**What we're doing:** deploy the petshop (an API plus four client workloads) into one namespace and enrol it in the mesh.

**How:** a single `istio.io/dataplane-mode=ambient` label on the namespace, no restarts and no sidecars.

**What you'll see:** the five pods running, ready for the identity walk-through.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

kubectl --context $CTX apply -f yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 2.2 · The identity is the certificate

**What we're doing:** show that a workload's identity is its mTLS certificate, issued from its ServiceAccount, and that this is what policy matches on.

**How:** list the leaf certificates ztunnel holds, next to the pods.

**What you'll see:** five pods but only **four** certificates. `checkout-blue` and `checkout-green` never appear by name because they share `sa/checkout`, so ztunnel presents one certificate for both. To the mesh they are the same caller. That is the ceiling §2.5 runs into and §2.7 removes.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


## 2.3 · Authorise on identity, at L4

**What we're doing:** allow only the `storefront` workload to reach `petstore`, and let ztunnel deny everything else, with no waypoint and no app change.

**How:** one `AuthorizationPolicy` that selects `petstore` and permits the `storefront` identity. ztunnel fails closed: once any ALLOW selects a workload, everything not named is denied. Principals use the `mesh1` trust domain, so a `cluster.local/...` principal here would match nothing.

**What you'll see:** `storefront` gets `200`, while `analytics` and both `checkout` pods get `000` (connection refused), decided purely on identity.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


**Watching in the Gloo UI and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry** over a rolling window — those edges are pre-policy 200s still ageing out. A workload denied at L4 never completes a request, so once its history rolls out of the window it draws nothing. Drop the Graph's time-interval picker to the smallest window to make it reshape within a minute or two. The per-connection allow/deny verdicts are in ztunnel's access logs — next.

## 2.4 · Read the L4 access logs

**What we're doing:** show that the allow/deny decisions are auditable, tagged with the caller's identity, at L4 with no waypoint.

**How:** read ztunnel's access log, which records every connection with the peer SPIFFE identities and the outcome.

**What you'll see:** `storefront` as **ALLOW** and `analytics` as **DENY**, each line carrying its `src.identity`.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'


## 2.5 · The limit of service-account identity

**What we're doing:** show the one thing this model cannot do: tell apart two pods that share a ServiceAccount.

**How:** add `sa/checkout` to the allowed set and call from both checkout pods.

**What you'll see:** both `checkout-blue` and `checkout-green` get in, and there is no L4 rule that admits one and blocks the other. They share `sa/checkout`, present the same certificate, and to ztunnel they are the same caller.

Why it matters for the customer: the everyday version is worse than blue/green. Any pod that never sets a ServiceAccount runs as the namespace `default`, so an ALLOW written for "the payments app" quietly admits every workload in that namespace. The two fixes are to give every workload its own ServiceAccount (as this app does) and, where pods genuinely share one, to close the gap with **workload claims** in §2.7.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000


## 2.6 · What else L4 can decide: namespace, conditions, DENY

**What we're doing:** show that identity is not the only thing ztunnel can authorise on. It also decides on source namespace, source IP, destination port and SNI, directly or in a CEL `when` clause, and a `DENY` always beats an `ALLOW`.

**How:** add a second caller in its own `warehouse` namespace, then walk it through allow, block and an explicit deny.

**What you'll see:** the warehouse caller allowed when the policy admits its namespace, then blocked when we narrow it, and finally an explicit `DENY` overriding an `ALLOW`. You can watch it appear and disappear in the Gloo UI Graph.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# A second client, in ANOTHER namespace — its identity is spiffe://mesh1/ns/warehouse/sa/warehouse-svc
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s

# ── verify: warehouse client is up and its identity is in the warehouse trust domain ──
kubectl --context $CTX -n warehouse get pods
echo "no ALLOW admits it yet, so it is denied for now:"
sleep 6; kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE ALLOW admits callers from TWO namespaces — decided by a CEL `when` on source.namespace
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop", "warehouse"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"
# both 200 — refresh the Graph: warehouse appears, serving petstore


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# Now BLOCK it — same policy, one value removed. warehouse fails closed.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"

# the deny as ztunnel logged it — FAIL-CLOSED: allow policies exist, none matched
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep warehouse | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'


**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW admits it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy. From the client both denials are the same `000` — the difference is in ztunnel's log: the warehouse deny read `allow policies exist, but none allowed` (fail-closed); this one reads `explicitly denied by: petshop/l4-deny-analytics` — the log **names the policy** that fired.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done

for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep analytics | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# reset the L4-surface policies before closing the gap
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found

# ── verify: only the SA-wide allow-storefront + allow-checkout remain ──
kubectl --context $CTX -n petshop get authorizationpolicy


## 2.7 · Close the gap with workload claims

**What we're doing:** solve the §2.5 problem, telling apart two pods on one ServiceAccount, still at L4 and still with no waypoint.

**How:** turn on `ENABLE_WORKLOAD_CLAIMS`. ztunnel then requests a certificate **per pod** instead of per ServiceAccount, istiod embeds signed claims in each one at issuance, and an `AuthorizationPolicy` matches those claims with CEL. On the 1.30 line this is one Helm value on ztunnel, same chart and version, all the multicluster values kept.

**What you'll see (over the next few cells):** every pod now holding its own certificate, `checkout-blue` carrying `tier: gold` and `checkout-green` carrying `tier: silver` in that certificate, and a claims-based policy that finally lets blue in and keeps green out.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE new value: ENABLE_WORKLOAD_CLAIMS. Everything else identical to the standup.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s

# ── verify: the flag is live on ztunnel ──
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel \
  -o jsonpath='{.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].name}={.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].value}{"\n"}'


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# every workload now holds a PER-POD cert (contrast §2.2: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"


**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5

# ── verify: the tier annotation is on each pod (istiod bakes it into the cert) ──
kubectl --context $CTX -n petshop get pod -l app=checkout \
  -o custom-columns=POD:.metadata.name,TIER:'.metadata.annotations.solo\.io\.security-claims/tier'


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# open the certs with openssl and read the claims off the SAN. Each cert gains an
# otherName SAN under Solo's OID (1.3.6.1.4.1.65865.1.1) whose payload is base64url JSON.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem
  echo "── $pod ──────────────────────────────────────────────"
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -A1 "Subject Alternative Name" | tail -1
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); print(json.dumps(json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))), indent=2))'
done


Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. Now authorize on it: swap the SA-wide checkout ALLOW for a claims-scoped one — the same `when` CEL shape as §2.6, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). Fail-closed: a workload with no claim never matches the ALLOW.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX -n petshop delete authorizationpolicy allow-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - mesh1/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge
for i in $(seq 1 24); do
  B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  echo "  waiting for convergence ($i/24): blue=$B green=$G (want blue=200, green!=200)"
  [ "$B" = "200" ] && [ "$G" != "200" ] && break; sleep 5
done
if [ "$G" = "200" ]; then
  echo "!! green is still allowed — is another ALLOW policy admitting it? kubectl -n petshop get authorizationpolicy"
fi
echo
echo "checkout-blue  (tier gold)   -> $B"
echo "checkout-green (tier silver) -> ${G:-DENIED}   (same sa/checkout, told apart by its cert claim)"


`checkout-blue -> 200`, `checkout-green -> denied` — same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §2.5 could not do. Note the distinction from what comes next: **this** CEL is over claims in the workload *certificate* at L4; §2.9's CEL is over the user's *request token* at the L7 waypoint.

## 2.8 · Add a waypoint for L7

**What we're doing:** move up to HTTP-level control (methods, paths, JWTs, rate limits), which L4 cannot see.

**How:** ambient adds a **waypoint**, and on Solo Enterprise the waypoint is **agentgateway** (class `enterprise-agentgateway-waypoint`), not Envoy. The split of work stays clean: ztunnel keeps proving the caller's identity at L4, then hands the connection to agentgateway for L7, and that proven identity rides along as `source.identity.*` for every policy. `setup.sh` already installed the control plane, so here we just create the waypoint and enrol the namespace.

**What you'll see:** the waypoint pod come up and its own request log showing the petshop traffic now flowing through it. We reset the L4 policies first so the L7 story stands on its own (in production you would keep both, defence in depth).

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# reset the L4 policies for a clean L7 story (incl. the claims policy)
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout --ignore-not-found

# the waypoint: a Gateway of class enterprise-agentgateway-waypoint …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petshop-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
# … then enrol the whole petshop NAMESPACE onto it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petshop-waypoint --overwrite
sleep 5
kubectl --context $CTX -n petshop rollout status deploy/petshop-waypoint --timeout=150s
# proof it is agentgateway: watch its own request log as the loops flow through
kubectl --context $CTX -n petshop logs deploy/petshop-waypoint --tail=2


## 2.9 · Authorise on the user's JWT, at L7

**What we're doing:** add user-level authorisation on top of workload identity: any signed-in user can read, only an admin can delete.

**How:** Keycloak (from `setup.sh`, realm `petshop`, users **alice**/`user` and **bob**/`admin`) issues the tokens. Two `EnterpriseAgentgatewayPolicy` objects sit on the waypoint. `jwtAuthentication` in Strict mode validates every request against Keycloak's JWKS, so no token means `401`. `authorization` then decides with CEL over the claims: any valid token may `GET`, and `DELETE` additionally needs `admin` in `realm_access.roles`.

**What you'll see:** no token → `401`, alice `GET` → `200`, alice `DELETE` → `403`, bob `DELETE` → `200`.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=300s
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    jwtAuthentication:
      mode: Strict
      providers:
        - issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          jwks:
            remote:
              backendRef: { name: keycloak, namespace: keycloak, port: 8080 }
              jwksPath: /realms/petshop/protocol/openid-connect/certs
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-authz, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    authorization:
      action: Allow
      policy:
        matchExpressions:
          - 'request.method == "GET"'
          - 'request.method == "DELETE" && "admin" in jwt.realm_access.roles'
EOF
sleep 10

# ── verify: both policies are attached to the waypoint ──
kubectl --context $CTX -n petshop get enterpriseagentgatewaypolicy petshop-jwt petshop-authz


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# the matrix: no token, alice (user), bob (admin).
# agentgateway answers a MISSING/invalid token with 401 (authentication),
# and a valid token that fails the CEL with 403 (authorization).
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


**Look at the Graph now — the denials are RED.** An L7 denial is a real HTTP response, so the Graph can finally draw a block: red edges from every (token-less) client loop into `petshop-waypoint`. Contrast the L4 sections, where a denial is a reset connection and a block only ever showed as an edge going quiet. L4 deny = silence; L7 deny = red edge with a status code.

## 2.10 · Route at the waypoint: canary and header shift

**What we're doing:** show the same waypoint that checks the JWT also does traffic management, and that policy and routing compose.

**How:** attach an `HTTPRoute` whose parent is the petstore **Service** (the GAMMA pattern). Callers keep calling `petstore:8080`; the waypoint enforces a 90/10 canary to `petstore-v2` and a header shift, so `x-beta: true` goes straight to v2. The JWT policy from §2.9 still gates every request.

**What you'll see:** roughly 90/10 across v1 and v2 with a valid token, the header pinning v2, and no token still `401` on every path.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# petstore-v2: the same app (it reports served_by), the SAME ServiceAccount
# (routing does not change identity), its own Service.
kubectl --context $CTX apply -f yaml/50-l7/30-petstore-v2.yaml
kubectl --context $CTX -n petshop rollout status deploy/petstore-v2 --timeout=120s

kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: petstore-split
  namespace: petshop
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: petstore
      port: 8080
  rules:
    - matches:
        - headers:
            - name: x-beta
              value: "true"
      backendRefs:
        - name: petstore-v2
          port: 8080
    - backendRefs:
        - name: petstore
          port: 8080
          weight: 90
        - name: petstore-v2
          port: 8080
          weight: 10
EOF
kubectl --context $CTX -n petshop get httproute petstore-split \
  -o jsonpath='{.status.parents[0].conditions[?(@.type=="Accepted")].status}{" — accepted by Service/"}{.status.parents[0].parentRef.name}'; echo
sleep 10


In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# the proof: distribution, header shift, and the JWT policy still in force
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— 20 GETs with alice's token: the canary split —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 20); do curl -s -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— 5 GETs with x-beta: true -> always v2 —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 5); do curl -s -m5 -H 'Authorization: Bearer $ALICE' -H 'x-beta: true' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— no token, even on the beta route —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -o /dev/null -w 'no token -> %{http_code}\n' -m5 -H 'x-beta: true' http://petstore:8080/pets"


## 2.11 · Rate limit by workload identity

**What we're doing:** rate-limit one workload without touching another, even when both carry the same user's token. The user's token says who the **user** is; the certificate says who the **workload** is, and here they meet.

**How:** a rate limit whose CEL condition keys on `source.identity.serviceAccount`, the identity ztunnel proved at L4, enforced locally in the waypoint.

**What you'll see:** `storefront` allowed five times then `429`, while `checkout` presenting the same user JWT is untouched. The budget followed the workload's certificate, not the user's token.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# 5 requests/minute for the storefront IDENTITY, everyone else unlimited
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-ratelimit-storefront, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    rateLimit:
      conditional:
        - condition: 'source.identity.serviceAccount == "storefront"'
          policy:
            local:
              - requests: 5
                unit: Minutes
EOF
sleep 8

KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— storefront, 8 rapid GETs (alice's token) —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo

echo "— checkout-blue, 8 rapid GETs (the SAME token) —"
kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo


`storefront: 200 ×5 then 429 429 429` — `checkout-blue: 200 ×8`, on the **same token**. The rate budget followed the workload's certificate, not the user's JWT. That is the identity thesis of this whole part, applied to L7 traffic management.

## 2.R · Reset Part 2

**What we're doing:** undo Part 2 so it can be re-run cleanly.

**How:** remove the petshop policies, waypoint, routes and v2, drop the claim annotations, and revert ztunnel to claims-off so §2.2's four-certificates moment works again. The petshop app itself stays deployed.

**What you'll see:** the petshop namespace clean of policies, routes and the waypoint, and the claims flag back off.

For a full "square one" wipe of the whole demo (both parts), use `./demo-scripts/reset.sh` instead.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX -n petshop delete enterpriseagentgatewaypolicy petshop-jwt petshop-authz petshop-ratelimit-storefront --ignore-not-found
kubectl --context $CTX -n petshop delete httproute petstore-split --ignore-not-found
kubectl --context $CTX -n petshop delete deploy petstore-v2 --ignore-not-found; kubectl --context $CTX -n petshop delete svc petstore-v2 --ignore-not-found
kubectl --context $CTX label namespace petshop istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CTX -n petshop delete gateway petshop-waypoint --ignore-not-found
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX delete ns warehouse --ignore-not-found
kubectl --context $CTX -n petshop patch deploy checkout-blue  --type=json -p='[{"op":"remove","path":"/spec/template/metadata/annotations/solo.io.security-claims~1tier"}]' 2>/dev/null || true
kubectl --context $CTX -n petshop patch deploy checkout-green --type=json -p='[{"op":"remove","path":"/spec/template/metadata/annotations/solo.io.security-claims~1tier"}]' 2>/dev/null || true

# ztunnel back to claims-off (same values as the standup)
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s
echo "Part 2 reset."

# ── verify: no petshop policies/routes/waypoint left; claims flag back off ──
kubectl --context $CTX -n petshop get authorizationpolicy,enterpriseagentgatewaypolicy,httproute,gateway 2>/dev/null | grep -v NAME || echo "  (petshop clean — app still deployed)"
echo "claims flag: $(kubectl --context $CTX -n $ISTIO_NS get ds ztunnel -o jsonpath='{.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].value}' 2>/dev/null || echo off)"
